# 04 — PPO Training on IoTNetworkEnv

This notebook trains a PPO agent in a custom Gymnasium environment for IoT network optimization.

In [ ]:
# Cell 1: Imports and project path setup
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from backend.environment.iot_network_env import IoTNetworkEnv, DEFAULT_CONFIG
from backend.environment.network_simulator import NetworkSimulator

In [ ]:
# Cell 2: Environment creation and compatibility check
env = IoTNetworkEnv(config=DEFAULT_CONFIG)
check_env(env, warn=True)
obs, info = env.reset(seed=42)
print('Initial observation shape:', obs.shape)
print('Initial info:', info)

In [ ]:
# Cell 3: Run random policy baseline
simulator = NetworkSimulator(env=env)
result = simulator.run_random_policy(n_steps=200, seed=7)
print('Baseline steps:', len(result.rewards))
print('Baseline mean reward:', float(np.mean(result.rewards)))

In [ ]:
# Cell 4: Train PPO agent
train_env = IoTNetworkEnv(config=DEFAULT_CONFIG)
model = PPO(
    'MlpPolicy',
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
)
model.learn(total_timesteps=20000)

In [ ]:
# Cell 5: Evaluate trained policy
eval_env = IoTNetworkEnv(config=DEFAULT_CONFIG)
obs, _ = eval_env.reset(seed=123)
episode_rewards = []
terminated = False
truncated = False
while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = eval_env.step(action)
    episode_rewards.append(float(reward))

print('Evaluation length:', len(episode_rewards))
print('Evaluation average reward:', float(np.mean(episode_rewards)))

In [ ]:
# Cell 6: Plot evaluation rewards
plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, label='Reward per step')
plt.title('PPO Evaluation Rewards')
plt.xlabel('Step')
plt.ylabel('Reward')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Save trained PPO model
weights_dir = project_root / 'backend' / 'models' / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)
model_path = weights_dir / 'ppo_iot.zip'
model.save(str(model_path))
print('Saved PPO model to:', model_path)